# 选修E3 · Day 1 上机：Transformer 架构与训练流程

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **tiktoken** 和 **transformers AutoTokenizer** 对营销文案做真实 tokenization，对比中英文 token 消耗
2. 从 **transformers AutoConfig** 读取 GPT-2 架构参数，推算参数量（~124M）
3. 用 **torch** 手写 Self-Attention 和 Transformer Block，理解架构而非黑箱
4. 在营销文案 token 上运行注意力，可视化注意力矩阵
5. 理解 CLM 预训练任务和训练三阶段（Pre-training/SFT/Alignment）

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：transformers（config+tokenizer）+ torch（手写注意力）+ tiktoken（BPE 分词）。
**不加载预训练权重**（避免下载 500MB+ 模型），仅用 config + tokenizer 做架构分析。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ transformers 仅用 config + tokenizer（秒级加载），不下载模型权重。
> tiktoken 和 torch 是纯本地库，无需 API key。

In [ ]:
# !pip install transformers torch tiktoken -q

## 1. 场景背景与营销映射

**核心命题**：LLM 是营销 Agent 的引擎。理解 Transformer 如何生成营销文案、token 成本如何影响推理成本。

**本上机解决**：
- 营销文案的中英文 token 消耗差异有多大？（直接影响推理成本）
- GPT-2 的架构参数有哪些？模型有多大？（理解 Scale Law）
- Self-Attention 如何关联营销文案中的关键词？（理解生成过程）

| TODO | 任务 | 真实库 | 营销映射 |
|------|------|--------|---------|
| TODO1 | Tokenization 对比 | tiktoken + transformers | 营销文案 token 成本 |
| TODO2 | GPT-2 架构分析 | transformers AutoConfig | 模型规模理解 |
| TODO3 | 手写 Self-Attention | torch | 注意力机制理解 |
| TODO4 | Multi-Head + TransformerBlock | torch | 完整架构块 |
| TODO5 | 注意力可视化 | torch + tiktoken | 营销关键词关联 |
| TODO6 | CLM 前向传播 + 训练阶段 | torch | 预训练任务理解 |

In [ ]:
import torch
import torch.nn.functional as F
import math
import tiktoken
from transformers import AutoConfig, AutoTokenizer

# ============================================================
# 营销文案语料（真实数据，基于电商场景）
# ============================================================

MARKETING_TEXTS = {
    "en_brief": "Write a Xiaohongshu marketing copy for a niacinamide serum targeting women aged 25-35",
    "zh_brief": "为一款烟酰胺精华液写小红书种草文案，目标人群25-35岁女性",
    "en_product": "Niacinamide Brightening Serum 5 percent niacinamide brightens skin tone and minimizes pores",
    "zh_product": "烟酰胺亮肤精华液 5%烟酰胺 提亮肤色 收缩毛孔",
}

# 模型定价表（$/token，基于 OpenAI 2026 定价）
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50e-6, "output": 10.00e-6},
    "gpt-4o-mini": {"input": 0.15e-6, "output": 0.60e-6},
}

print("环境初始化完成")
print(f"transformers + torch + tiktoken 已加载")
print(f"营销文案样本: {len(MARKETING_TEXTS)} 条")
print(f"模型定价: {list(MODEL_PRICING.keys())}")

## TODO 1：用 tiktoken + transformers 对营销文案做 Tokenization

**tiktoken** 是 OpenAI 的 BPE 分词器，**transformers AutoTokenizer** 是 HuggingFace 的分词器接口。

**关键对比**：
- 英文 ~1 token ≈ 0.75 单词
- 中文 1 汉字 ≈ 1-2 token（BPE 主要在英文数据上训练）
- 这直接影响 LLM API 的 token 计费和推理成本

**BPE 子词可视化**：`enc.decode([token_id])` 可查看每个子词 token 的文本。高频词是完整 token，低频词被拆分为子词。

In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 用 tiktoken.get_encoding('gpt2') 和 AutoTokenizer.from_pretrained('gpt2') 加载分词器
# 2) 对 4 条营销文案分别编码，对比中英文 token 数
# 3) 用 enc.decode([token_id]) 可视化 BPE 子词拆分
# 4) 计算 zh_brief + 一段营销文案响应的 token 成本
# 提示: enc_tiktoken = tiktoken.get_encoding('gpt2')
#       tok_hf = AutoTokenizer.from_pretrained('gpt2')
#       enc_tiktoken.encode(text) 返回 token 列表
#       MODEL_PRICING 已定义，cost = tokens * rate
raise NotImplementedError("请实现 tokenization 对比分析")
# ====================

## TODO 2：从 GPT-2 Config 读取架构参数，推算参数量

**AutoConfig.from_pretrained("gpt2")** 秒级加载 GPT-2 架构参数（不下载权重）：
- `n_layer`：Transformer Block 层数
- `n_head`：多头注意力的头数
- `n_embd`：嵌入维度
- `vocab_size`：词表大小
- `n_positions`：最大序列长度

**参数量推算**：
- Token embedding：`vocab_size × n_embd`
- Position embedding：`n_positions × n_embd`
- 每个 Block：attention（4 × n_embd²）+ FFN（8 × n_embd²）+ biases
- 总计 ≈ 124M（GPT-2 small）

**Scale Law 启示**：模型性能随参数量、数据量、计算量可预测地提升。

In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 用 AutoConfig.from_pretrained('gpt2') 加载 GPT-2 架构参数
# 2) 打印 n_layer, n_head, n_embd, vocab_size, n_positions
# 3) 推算参数量：embedding + n_layer * per_block + final_ln
#    - Token embedding = vocab_size * n_embd
#    - Position embedding = n_positions * n_embd
#    - 每个 Block ≈ 4*n_embd^2 (attn) + 8*n_embd^2 (FFN) + biases
# 提示: config = AutoConfig.from_pretrained('gpt2')
#       config.n_layer, config.n_head, config.n_embd, config.vocab_size
raise NotImplementedError("请实现 GPT-2 架构分析")
# ====================

## TODO 3：手写 Self-Attention

**Self-Attention 公式**：

```
Attention(Q, K, V) = softmax(Q × K^T / √d_k) × V
```

**计算步骤**：
1. Q = x @ W_q, K = x @ W_k, V = x @ W_v（线性投影）
2. scores = Q @ K^T / √d_k（点积 + 缩放）
3. attn = softmax(scores)（归一化为概率分布，行和为 1）
4. out = attn @ V（加权聚合 Value）

**为什么除以 √d_k**：防止点积值过大导致 softmax 梯度消失。

In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 定义 self_attention(x, W_q, W_k, W_v) 函数
# 2) 计算 Q=x@W_q, K=x@W_k, V=x@W_v
# 3) scores = Q @ K^T / sqrt(d_k)
# 4) attn = softmax(scores, dim=-1)
# 5) out = attn @ V
# 提示: K.transpose(-2, -1) 转置最后两维
#       F.softmax(scores, dim=-1) 按行归一化
#       math.sqrt(d_k) 缩放
raise NotImplementedError("请实现 Self-Attention")
# ====================

## TODO 4：Multi-Head Attention + Transformer Block

**Multi-Head Attention**：将 Q/K/V 分成 `n_heads` 组，每组独立计算 Attention，然后拼接。

**Transformer Block 完整结构**：
```
输入
  ├─ Multi-Head Self-Attention
  |    └─ 残差连接 + LayerNorm
  ├─ Feed-Forward Network (FFN, 4x 扩展)
  |    └─ 残差连接 + LayerNorm
输出
```

- **残差连接**：`x = LayerNorm(x + f(x))`，解决深层网络梯度消失
- **FFN**：`Linear(d, 4d) → GELU → Linear(4d, d)`，非线性变换
- **LayerNorm**：归一化每层输出，稳定训练

In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 定义 TransformerBlock(d, n_heads) 继承 torch.nn.Module
# 2) __init__: W_q/W_k/W_v/W_o (Linear), ffn (Linear->GELU->Linear), ln1/ln2 (LayerNorm)
# 3) forward: 多头注意力 -> 残差+ln1 -> FFN -> 残差+ln2
# 4) 多头: view(B,S,n_heads,d_h).transpose(1,2) 重塑为 (B,heads,S,d_h)
# 提示: d_h = D // n_heads
#       FFN: Linear(d, d*4) -> GELU -> Linear(d*4, d)
#       残差: x = self.ln1(x + attn_out)
raise NotImplementedError("请实现 TransformerBlock")
# ====================

## TODO 5：在营销文案 Token 上可视化注意力

将营销文案 token化后，用 TODO3 的 Self-Attention 计算注意力矩阵，查看哪些词相互关联最强。

**步骤**：
1. 用 tiktoken 将英文产品描述 token 化
2. 为每个 token 生成随机 embedding（教学版，真实模型用学习的 embedding）
3. 运行 Self-Attention
4. 打印注意力矩阵，找出每个 token 最关注的其他 token

**营销洞察**：注意力矩阵显示模型在生成文案时"看"哪些词。例如"niacinamide"可能强烈关注"serum"和"brightens"。

In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 用 tiktoken 将 MARKETING_TEXTS['en_product'] token 化
# 2) 为每个 token 生成随机 embedding (torch.randn(seq, d))
# 3) 调用 self_attention() 计算注意力矩阵
# 4) 打印注意力矩阵（seq x seq）
# 5) 用 attn[i].topk(2) 找每个 token 最关注的 2 个 token
# 提示: enc = tiktoken.get_encoding('gpt2')
#       tokens = enc.encode(text)
#       subwords = [enc.decode([t]) for t in tokens]
#       调用 TODO3 的 self_attention(x, W_q, W_k, W_v)
raise NotImplementedError("请实现注意力可视化")
# ====================

## TODO 6：CLM 前向传播 + 训练三阶段

**CLM（Causal Language Modeling）**：预训练的核心任务--给定前面的 token，预测下一个 token。

```
输入:  [t0, t1, t2, ..., t_{n-1}]
目标:  [t1, t2, t3, ..., t_n]
```

**实现 MiniGPT**：Token Embedding + Position Embedding + N 个 TransformerBlock + 输出投影。

**训练三阶段**：
1. **Pre-training**：海量文本预测下一 token → Base Model
2. **SFT**：指令-回答对监督微调 → Chat Model
3. **Alignment**：RLHF/DPO 对齐人类偏好 → Aligned Model

本 TODO 演示 CLM 前向传播（不训练，仅展示 logits 和 loss 计算方式）。

In [ ]:
# ===== 你的代码 =====
# TODO: 你的代码
# 1) 定义 MiniGPT(vocab_size, d, n_heads, n_layers) 继承 torch.nn.Module
# 2) __init__: tok_emb + pos_emb + n_layers个TransformerBlock + ln + head
# 3) forward: x = tok_emb(idx) + pos_emb(pos) -> blocks -> ln -> head -> logits
# 4) 用营销文案 tokens 做前向传播，打印 logits shape
# 5) 计算 CLM loss: shift_logits[:, :-1] vs targets[:, 1:]
# 6) 打印训练三阶段概述
# 提示: TransformerBlock 来自 TODO4
#       F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
raise NotImplementedError("请实现 MiniGPT + CLM 前向传播")
# ====================

## 3. 反思与前沿

### 反思问题
1. 营销文案的中英文 token 消耗差异有多大？对推理成本有什么影响？
2. GPT-2 small 有 124M 参数，GPT-3 有 175B--参数量增长 1000 倍带来什么能力提升？推理成本呢？
3. Self-Attention 的注意力矩阵中，对角线值通常较大，为什么？（每个 token 最关注自己）
4. 预训练用"预测下一 token"这么简单的任务，为什么能训练出如此强大的模型？

### 2026 前沿：推理成本优化
- **DeepSeek-MoE**（arXiv 2401.04088）：MoE 架构，671B 总参数 / 37B 激活参数，推理成本远低于 Dense 模型
- **投机解码**（arXiv 2211.17192）：小模型生成候选 token，大模型并行验证，延迟降低 2-3x
- **vLLM**（https://github.com/vllm-project/vllm）：PagedAttention + 连续批处理，吞吐量 14-24x
- **多模态 + 对比学习**：CLIP 用对比损失对齐图文，GPT-4o 端到端多模态训练

参考 [DeepSeek-MoE](https://arxiv.org/abs/2401.04088) + [投机解码论文](https://arxiv.org/abs/2211.17192) + [vLLM](https://github.com/vllm-project/vllm)。